# Context Managers

A **context manager** is an object that manages setup and teardown of a resource.

You use them with the `with` statement:
```python
with open('file.txt') as f:
    data = f.read()
# file is automatically closed here, even if an exception occurred
```

Context managers guarantee that cleanup code always runs — great for:
- File handles
- Database connections
- Locks (threading)
- Timing blocks of code
- Temporarily changing settings

## 1. The Built-in `with` Statement

In [ ]:
# Without context manager — easy to forget cleanup
f = open('/tmp/test.txt', 'w')
try:
    f.write('Hello, World!')
finally:
    f.close()  # must remember this!

# With context manager — automatic cleanup
with open('/tmp/test.txt', 'w') as f:
    f.write('Hello, World!')
# f.close() called automatically

# Verify it's closed
print(f'File closed: {f.closed}')

## 2. Class-Based Context Manager: `__enter__` and `__exit__`

Implement a context manager by defining `__enter__` and `__exit__` on a class.

In [ ]:
import time

class Timer:
    """Measure elapsed time for a block of code."""
    
    def __enter__(self):
        self._start = time.perf_counter()
        return self  # this is bound to the 'as' variable
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self._start
        print(f'Elapsed: {self.elapsed:.6f}s')
        return False  # False = don't suppress exceptions


with Timer() as t:
    result = sum(range(1_000_000))

print(f'Sum: {result}')
print(f'Access elapsed: {t.elapsed:.6f}s')

In [ ]:
# __exit__ receives exception info — you can handle or suppress it
class SuppressErrors:
    """Suppress specific exception types."""
    
    def __init__(self, *exception_types):
        self.exceptions = exception_types
    
    def __enter__(self):
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        # Return True to suppress the exception
        if exc_type and issubclass(exc_type, self.exceptions):
            print(f'Suppressed {exc_type.__name__}: {exc_val}')
            return True
        return False  # re-raise other exceptions


with SuppressErrors(ZeroDivisionError, ValueError):
    print(1 / 0)  # ZeroDivisionError — suppressed

print('Execution continues after suppressed error')

## 3. Function-Based Context Manager with `contextlib`

`@contextmanager` lets you write a context manager as a generator function — much simpler!

In [ ]:
from contextlib import contextmanager

@contextmanager
def timer(label=''):
    """Measure time for a block."""
    import time
    start = time.perf_counter()
    try:
        yield  # code inside 'with' block runs here
    finally:
        elapsed = time.perf_counter() - start
        print(f'{label}: {elapsed:.6f}s')


with timer('List comprehension'):
    result = [x**2 for x in range(100_000)]

with timer('Generator expression'):
    result = sum(x**2 for x in range(100_000))

In [ ]:
from contextlib import contextmanager
import os

@contextmanager
def working_directory(path):
    """Temporarily change the working directory."""
    original = os.getcwd()
    try:
        os.chdir(path)
        yield
    finally:
        os.chdir(original)  # always restore, even on exception


print(f'Before: {os.getcwd()}')
with working_directory('/tmp'):
    print(f'Inside: {os.getcwd()}')
print(f'After: {os.getcwd()}')

## 4. `contextlib` Utilities

In [ ]:
from contextlib import suppress, nullcontext

# suppress — ignore specific exceptions cleanly
with suppress(FileNotFoundError):
    with open('/tmp/nonexistent_file.txt') as f:
        data = f.read()
print('No error — FileNotFoundError was suppressed')

# nullcontext — a do-nothing context manager (useful for optional context managers)
def process(data, debug=False):
    ctx = timer('processing') if debug else nullcontext()
    with ctx:
        return [x * 2 for x in data]

result = process([1, 2, 3], debug=False)  # no timing
print(result)

## 5. Nested Context Managers

In [ ]:
# Old way — nested
with open('/tmp/source.txt', 'w') as src:
    src.write('source content')

with open('/tmp/source.txt', 'r') as src:
    with open('/tmp/dest.txt', 'w') as dest:
        dest.write(src.read().upper())

# Better — multiple context managers on one line
with open('/tmp/source.txt', 'r') as src, open('/tmp/dest2.txt', 'w') as dest:
    dest.write(src.read().upper())

# Verify
with open('/tmp/dest2.txt') as f:
    print(f.read())

## Common Mistakes

### Mistake 1: Returning `True` From `__exit__` Unintentionally Suppresses Exceptions

In [ ]:
class BadContextManager:
    def __enter__(self): return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        print('Cleanup done')
        return True  # BAD: accidentally suppresses ALL exceptions!

class GoodContextManager:
    def __enter__(self): return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        print('Cleanup done')
        return False  # GOOD: re-raises exceptions (or None is also fine)

with BadContextManager():
    raise ValueError('This error is silently eaten!')
print('BadContextManager suppressed the error silently')

## Exercises

### Exercise 1 (Easy): File Write Context Manager
Write a context manager that opens a file, writes content, and ensures the file is closed (even on error).

In [ ]:
# Your solution here


In [ ]:
# Solution
from contextlib import contextmanager

@contextmanager
def write_file(path, mode='w'):
    f = open(path, mode)
    try:
        yield f
    finally:
        f.close()
        print(f'File {path} closed.')

with write_file('/tmp/output.txt') as f:
    f.write('Line 1\n')
    f.write('Line 2\n')

with open('/tmp/output.txt') as f:
    print(f.read())

## Mini-Projects: Timer + File Transaction

In [ ]:
import time
from contextlib import contextmanager

class Timer:
    """Reusable timer context manager."""
    
    def __init__(self, label=''):
        self.label = label
        self.elapsed = None
    
    def __enter__(self):
        self._start = time.perf_counter()
        return self
    
    def __exit__(self, *args):
        self.elapsed = time.perf_counter() - self._start
        label = f'{self.label}: ' if self.label else ''
        print(f'{label}{self.elapsed:.6f}s')


@contextmanager  
def atomic_write(filepath):
    """
    Write to a temp file, then rename to final path.
    If an error occurs, the original file is not touched.
    """
    import os
    tmp_path = filepath + '.tmp'
    try:
        with open(tmp_path, 'w') as f:
            yield f
        os.replace(tmp_path, filepath)  # atomic on most filesystems
        print(f'Successfully written to {filepath}')
    except Exception as e:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        print(f'Write failed, original file untouched: {e}')
        raise


# Test Timer
with Timer('Sorting 10k items'):
    import random
    data = random.sample(range(100_000), 10_000)
    sorted_data = sorted(data)

# Test atomic write
with atomic_write('/tmp/important.txt') as f:
    f.write('This write is atomic!\n')
    f.write('Either all is written or nothing is.')

In [ ]:
# Real-world: database-like transaction context manager

class InMemoryDB:
    """Simple in-memory store with transaction support."""
    
    def __init__(self):
        self._data = {}
        self._backup = None
    
    def set(self, key, value):
        self._data[key] = value
    
    def get(self, key):
        return self._data.get(key)
    
    def __enter__(self):
        self._backup = dict(self._data)  # snapshot
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None:
            # Exception occurred — rollback
            self._data = self._backup
            print(f'Transaction rolled back due to: {exc_val}')
            return True  # suppress exception
        print('Transaction committed')


db = InMemoryDB()
db.set('x', 10)

# Successful transaction
with db:
    db.set('y', 20)
    db.set('z', 30)
print('After commit:', db._data)

# Failed transaction — rolled back
with db:
    db.set('y', 999)
    raise ValueError('Something went wrong!')
print('After rollback:', db._data)  # y is still 20